# Capital Gains Tax (CGT) Calculator

### Overview
This notebook automates the calculation of **CGT** for Australian shares.

It imports trade history from a CSV export, tracks individual purchase parcels, applies optimal parcel selection strategies (Highest-Cost-First) and automatically applies the **50% CGT discount** for holdings held over 12 months (365 days).

---

### Key features
* Automatic CSV import; Reads buy and sell confirmations directly from broker exports.
* Chronological auto-sorting: Trades are processed in exact historical order regardless of CSV row order.
* Tax minimisation (HIFO) strategy: Allocates sales against highest cost base parcels first to reduce taxable capital gains.
* ATO myTax ready output: Prints final totals mapped directly to the **Total Current Year Capital Gains** and **Net Capital Gains** fields on myTax.

### Step 1: Define the 'Parcel' Data Structure

A **Parcel** represents a specific batch of units bought on a specific date.
Under ATO TD 33 rules, individual parcels must be tracked separately so their specific cost base and holding duration can be evaluated when sold.

In [4]:
import csv
import os
from dataclasses import dataclass
from datetime import date, datetime
from IPython.display import HTML, display

# =========================================================================
# USER INPUT: ENTER YOUR FILE NAME HERE
# =========================================================================
FILE_NAME = "Trade Confirmation_SAMPLE.csv"


@dataclass
class Parcel:
    parcel_id: int
    ticker: str
    buy_date: date
    remaining_units: float
    current_cost_base: float

    @property
    def unit_cost_base(self) -> float:
        if self.remaining_units <= 0:
            return 0.0
        return self.current_cost_base / self.remaining_units

### Step 2: The 'TaxTracker' Engine

The 'TaxTracker' class handles the core logic:
1. Reads and cleans trade history from the CSV.
2. Sorts all trades chronologically from oldest to newest.
3. Logs buys into individual parcels with brokerage included in the cost base.
4. Processes sells using the HIFO strategy to minimise tax.
5. Applies CGT discounts (50%) for parcels held for at least 12 months.

In [8]:
class TaxTracker:

    def __init__(self):
        self.parcels: list[Parcel] = []
        self._next_id = 1
        self.total_gross_gains_fy = 0.0
        self.total_net_taxable_gains_fy = 0.0
        self.total_proceeds_fy = 0.0
        self.sales_log = []
        self.skipped_rows = []

    def _parse_date(self, date_str: str) -> date:
        date_str = date_str.strip()
        for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"):
            try:
                return datetime.strptime(date_str, fmt).date()
            except ValueError:
                pass
        parts = date_str.split("/")
        if len(parts) == 3:
            return date(int(parts[2]), int(parts[1]), int(parts[0]))
        raise ValueError(f"Could not parse date: {date_str}")

    def import_trades_from_csv(
        self, file_path: str = FILE_NAME, col_map: dict = None
    ):
        if col_map is None:
            col_map = {
                "ticker": "AsxCode",
                "date": "Trade Date",
                "type": "Order Type",
                "units": "Quantity",
                "price": "Price",
                "brokerage": "Brokerage",
            }

        raw_trades = []
        self.skipped_rows = []

        if not os.path.exists(file_path):
            self.skipped_rows.append({
                "line": "-",
                "reason": f"File not found: '{file_path}' in local directory.",
            })
            return

        try:
            with open(file_path, mode="r", encoding="utf-8-sig") as file:
                reader = csv.DictReader(file)
                for line_no, row in enumerate(reader, start=2):
                    clean_row = {
                        k.strip(): v.strip() for k, v in row.items() if k
                    }

                    try:
                        raw_trades.append({
                            "order_type": clean_row[col_map["type"]].upper(),
                            "ticker": clean_row[col_map["ticker"]],
                            "trade_date": self._parse_date(
                                clean_row[col_map["date"]]
                            ),
                            "units": float(clean_row[col_map["units"]]),
                            "price": float(clean_row[col_map["price"]]),
                            "brokerage": float(clean_row[col_map["brokerage"]]),
                        })
                    except KeyError as e:
                        self.skipped_rows.append({
                            "line": line_no,
                            "reason": f"Missing column header {e}",
                        })
                    except ValueError as e:
                        if "convert string to float" in str(e):
                            reason = "Empty or missing numeric value (Quantity/Price/Brokerage)"
                        else:
                            reason = f"Invalid date or number format ({e})"
                        self.skipped_rows.append(
                            {"line": line_no, "reason": reason}
                        )

        except Exception as e:
            self.skipped_rows.append(
                {"line": "-", "reason": f"Critical error reading file: {e}"}
            )
            return

        raw_trades.sort(key=lambda t: t["trade_date"])

        for trade in raw_trades:
            if "BUY" in trade["order_type"]:
                self.buy_holding(
                    ticker=trade["ticker"],
                    buy_date=trade["trade_date"],
                    units=trade["units"],
                    unit_price=trade["price"],
                    brokerage=trade["brokerage"],
                )
            elif "SELL" in trade["order_type"]:
                self.sell_holding(
                    ticker=trade["ticker"],
                    sale_date=trade["trade_date"],
                    units_to_sell=trade["units"],
                    sale_price=trade["price"],
                    brokerage=trade["brokerage"],
                )

    def buy_holding(
        self,
        ticker: str,
        buy_date: date,
        units: float,
        unit_price: float,
        brokerage: float,
    ):
        total_cost_base = (units * unit_price) + brokerage
        parcel = Parcel(
            parcel_id=self._next_id,
            ticker=ticker,
            buy_date=buy_date,
            remaining_units=units,
            current_cost_base=total_cost_base,
        )
        self.parcels.append(parcel)
        self._next_id += 1

    def sell_holding(
        self,
        ticker: str,
        sale_date: date,
        units_to_sell: float,
        sale_price: float,
        brokerage: float,
    ):
        gross_proceeds = (units_to_sell * sale_price) - brokerage

        active_parcels = [
            p
            for p in self.parcels
            if p.ticker == ticker
            and p.remaining_units > 0
            and p.buy_date <= sale_date
        ]
        total_available = sum(p.remaining_units for p in active_parcels)

        if total_available < units_to_sell:
            print(
                f"SALE EXECUTION ERROR: Insufficient Units for {ticker} "
                f"({units_to_sell:g} requested on {sale_date.strftime('%d/%m/%Y')}, "
                f"available: {total_available:g})"
            )
            return

        active_parcels.sort(key=lambda p: p.unit_cost_base, reverse=True)

        remaining_to_sell = units_to_sell
        trade_gross_gain = 0.0
        trade_taxable_gain = 0.0

        for parcel in active_parcels:
            if remaining_to_sell <= 0:
                break

            units_from_parcel = min(
                parcel.remaining_units, remaining_to_sell
            )
            fraction_of_parcel = units_from_parcel / parcel.remaining_units

            cost_base_allocated = (
                parcel.current_cost_base * fraction_of_parcel
            )
            proceeds_allocated = (
                units_from_parcel / units_to_sell
            ) * gross_proceeds
            gross_gain = proceeds_allocated - cost_base_allocated

            try:
                one_year_later = parcel.buy_date.replace(
                    year=parcel.buy_date.year + 1
                )
            except ValueError:
                one_year_later = parcel.buy_date.replace(
                    year=parcel.buy_date.year + 1, month=3, day=1
                )

            is_discounted = sale_date > one_year_later

            if gross_gain > 0 and is_discounted:
                taxable_gain = gross_gain * 0.5
            else:
                taxable_gain = gross_gain

            parcel.remaining_units -= units_from_parcel
            parcel.current_cost_base -= cost_base_allocated
            remaining_to_sell -= units_from_parcel

            trade_gross_gain += gross_gain
            trade_taxable_gain += taxable_gain

            self.sales_log.append({
                "ticker": ticker,
                "sale_date": sale_date,
                "parcel_id": parcel.parcel_id,
                "buy_date": parcel.buy_date,
                "units_sold": units_from_parcel,
                "discount_applied": is_discounted and gross_gain > 0,
                "gross_gain": gross_gain,
                "taxable_gain": taxable_gain,
            })

        self.total_proceeds_fy += gross_proceeds
        self.total_gross_gains_fy += trade_gross_gain
        self.total_net_taxable_gains_fy += trade_taxable_gain

    def display_html_summary(self):
        """Renders styled HTML dashboard with import alert support."""
        html_code = f"""
        <style>
            .dashboard-container {{
                font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
                max-width: 850px;
                margin: 0 auto;
                color: #1f2937;
            }}
            .metrics-grid {{
                display: flex;
                gap: 16px;
                margin-bottom: 24px;
            }}
            .metric-card {{
                flex: 1;
                background: #f9fafb;
                border: 1px solid #e5e7eb;
                border-radius: 10px;
                padding: 16px;
                text-align: center;
            }}
            .metric-label {{
                font-size: 13px;
                color: #6b7280;
                font-weight: 500;
                margin-bottom: 6px;
            }}
            .metric-value {{
                font-size: 22px;
                font-weight: 700;
                color: #111827;
            }}
            .metric-value.green {{ color: #16a34a; }}
            .sale-card {{
                background: #ffffff;
                border: 1px solid #e5e7eb;
                border-radius: 12px;
                padding: 20px;
                margin-bottom: 20px;
                box-shadow: 0 1px 3px rgba(0,0,0,0.04);
            }}
            .card-header {{
                display: flex;
                justify-content: space-between;
                align-items: center;
                border-bottom: 1px solid #f3f4f6;
                padding-bottom: 12px;
                margin-bottom: 16px;
            }}
            .ticker-badge {{
                background: #e0f2fe;
                color: #0369a1;
                font-weight: 700;
                padding: 4px 10px;
                border-radius: 6px;
                font-size: 14px;
            }}
            .discount-pill {{
                background: #dcfce7;
                color: #15803d;
                font-size: 11px;
                font-weight: 600;
                padding: 2px 8px;
                border-radius: 12px;
            }}
            .no-discount {{ color: #9ca3af; font-size: 12px; }}
            .tax-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 14px;
            }}
            .tax-table th {{
                text-align: left;
                color: #6b7280;
                font-weight: 600;
                padding-bottom: 10px;
                border-bottom: 2px solid #f3f4f6;
            }}
            .tax-table td {{
                padding: 10px 0;
                border-bottom: 1px solid #f9fafb;
            }}
            .pos {{ color: #16a34a; font-weight: 600; text-align: right; }}
            .neg {{ color: #dc2626; font-weight: 600; text-align: right; }}
        </style>

        <div class="dashboard-container">
            <h2 style="margin-bottom: 16px; font-size: 20px;">ATO myTax Lodgement Breakdown</h2>
            
            <div class="metrics-grid">
                <div class="metric-card">
                    <div class="metric-label">Total Proceeds</div>
                    <div class="metric-value">${self.total_proceeds_fy:,.2f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Gross Capital Gains</div>
                    <div class="metric-value">${self.total_gross_gains_fy:,.2f}</div>
                </div>
                <div class="metric-card">
                    <div class="metric-label">Net Taxable Gain</div>
                    <div class="metric-value green">${self.total_net_taxable_gains_fy:,.2f}</div>
                </div>
            </div>
        """

        if self.skipped_rows:
            html_code += f"""
            <div style="background: #ffffff; border: 1px solid #fecaca; border-radius: 12px; padding: 18px; margin-bottom: 24px;">
                <div style="color: #dc2626; font-weight: 700; font-size: 15px; margin-bottom: 12px;">
                    Import Alerts ({len(self.skipped_rows)} rows skipped)
                </div>
                <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
                    <thead>
                        <tr style="border-bottom: 2px solid #fee2e2; text-align: left; color: #991b1b;">
                            <th style="padding-bottom: 6px; width: 100px;">Line #</th>
                            <th style="padding-bottom: 6px;">Issue Detected</th>
                        </tr>
                    </thead>
                    <tbody>
            """
            for err in self.skipped_rows:
                line_str = (
                    f"Line {err['line']}"
                    if str(err["line"]).isdigit()
                    else err["line"]
                )
                html_code += f"""
                    <tr style="border-bottom: 1px solid #fef2f2;">
                        <td style="padding: 8px 0; color: #dc2626; font-weight: 600;">{line_str}</td>
                        <td style="padding: 8px 0; color: #4b5563;">{err['reason']}</td>
                    </tr>
                """
            html_code += "</tbody></table></div>"

        sales_by_event = {}
        for sale in self.sales_log:
            key = (sale["ticker"], sale["sale_date"])
            if key not in sales_by_event:
                sales_by_event[key] = []
            sales_by_event[key].append(sale)

        for (ticker, sale_date), parcels in sales_by_event.items():
            total_units = sum(p["units_sold"] for p in parcels)
            sale_date_str = sale_date.strftime("%d/%m/%Y")

            html_code += f"""
            <div class="sale-card">
                <div class="card-header">
                    <div>
                        <span class="ticker-badge">{ticker}</span>
                        <strong style="margin-left: 8px; font-size: 15px;">Sale of {total_units:.0f} units</strong>
                    </div>
                    <span style="color: #6b7280; font-size: 13px; font-weight: 500;">{sale_date_str}</span>
                </div>
                <table class="tax-table">
                    <thead>
                        <tr>
                            <th>Parcel</th>
                            <th>Buy Date</th>
                            <th>Discount Status</th>
                            <th style="text-align: right;">Gross Gain</th>
                            <th style="text-align: right;">Net Gain</th>
                        </tr>
                    </thead>
                    <tbody>
            """

            for p in parcels:
                gross_cls = "pos" if p["gross_gain"] >= 0 else "neg"
                taxable_cls = "pos" if p["taxable_gain"] >= 0 else "neg"

                discount_badge = (
                    '<span class="discount-pill">50% CGT Discount</span>'
                    if p["discount_applied"]
                    else '<span class="no-discount">—</span>'
                )

                html_code += f"""
                    <tr>
                        <td>Parcel #{p['parcel_id']} ({p['units_sold']:.0f} units)</td>
                        <td>{p['buy_date'].strftime('%d/%m/%Y')}</td>
                        <td>{discount_badge}</td>
                        <td class="{gross_cls}">${p['gross_gain']:,.2f}</td>
                        <td class="{taxable_cls}">${p['taxable_gain']:,.2f}</td>
                    </tr>
                """

            html_code += """
                    </tbody>
                </table>
            </div>
            """
            
        html_code += "</div>"
        display(HTML(html_code))

### Step 3: Run the Tax Analysis

Initialise the 'TaxTracker' and load consolidated CSV file.

In [9]:
tracker = TaxTracker()
tracker.import_trades_from_csv()
tracker.display_html_summary()

Parcel,Buy Date,Discount Status,Gross Gain,Net Gain
Parcel #4 (2 units),23/08/2022,—,$-31.31,$-31.31
Parcel #11 (1 units),02/01/2025,—,$22.23,$22.23
Parcel #12 (2 units),22/01/2025,—,$45.93,$45.93
Parcel #8 (1 units),10/07/2024,50% CGT Discount,$38.10,$19.05
Parcel #6 (8 units),28/07/2023,50% CGT Discount,$578.28,$289.14
Parcel,Buy Date,Discount Status,Gross Gain,Net Gain
Parcel #7 (5 units),26/06/2024,50% CGT Discount,$30.85,$15.42
Parcel #5 (28 units),28/07/2023,50% CGT Discount,$462.87,$231.43
Parcel #3 (6 units),23/08/2022,50% CGT Discount,$133.68,$66.84
